### 3.5.13. Penalty and Barrier Methods

$$
\textbf{exterior penalty:}\quad P_c(\mathbf{x}) = f(\mathbf{x}) + \frac{c}{2}\sum_i \max\big(0,\,g_i(\mathbf{x})\big)^2,
$$

$$
\textbf{interior barrier:}\quad B_\mu(\mathbf{x}) = f(\mathbf{x}) - \mu\sum_i \log\big(-g_i(\mathbf{x})\big).
$$

**Explanation:**

Penalty and barrier methods replace a constrained problem with a sequence of *unconstrained* ones, whose solutions trace a path to the constrained optimum. The exterior quadratic penalty charges a growing cost $c$ for constraint violation, so its minimizers approach the solution from the **infeasible** side as $c\to\infty$. The interior logarithmic barrier instead adds a term that diverges at the constraint boundary, so its minimizers stay strictly **feasible** and approach the solution from the inside as $\mu\to 0$. The barrier route is the foundation of [interior-point methods](../09_Algorithms/44_primal_dual_search_direction.ipynb); the barrier subproblem and its central path are studied in detail under the [logarithmic barrier function](../03_Convex_Optimization/50_logarithmic_barrier_function.ipynb).

**Numerical Example:**

$$
\min_x\; x^2 \quad\text{subject to}\quad x \ge 1,
$$
whose solution is $x^\star = 1$ with the constraint active.

**Penalty** ($g(x)=1-x$): for $x<1$, $P_c(x) = x^2 + \tfrac{c}{2}(1-x)^2$, minimized at
$$
x_c = \frac{c}{c+2} \;\xrightarrow[c\to\infty]{}\; 1^- ,
$$
so $x_2 = 0.5,\ x_{10}\approx 0.833,\ x_{100}\approx 0.980$ — approaching from below.

**Barrier**: $B_\mu(x) = x^2 - \mu\log(x-1)$, minimized where $2x = \mu/(x-1)$, i.e.
$$
x_\mu = \frac{1 + \sqrt{1 + 2\mu}}{2} \;\xrightarrow[\mu\to 0]{}\; 1^+ ,
$$
so $x_1\approx 1.366,\ x_{0.1}\approx 1.048,\ x_{0.01}\approx 1.005$ — approaching from above. The two paths meet at $x^\star=1$ from opposite sides.

In [1]:
import sympy as sp

x, c, mu = sp.symbols("x c mu", positive=True)

penalty_objective = x**2 + (c/2)*(1 - x)**2
penalty_minimizer = sp.solve(sp.diff(penalty_objective, x), x)[0]

barrier_objective = x**2 - mu*sp.log(x - 1)
feasible_roots = [root for root in sp.solve(sp.diff(barrier_objective, x), x) if root.subs(mu, 1) > 1]
barrier_minimizer = feasible_roots[0]

print("penalty minimizer x_c =", penalty_minimizer)
print("  values:", {value: float(penalty_minimizer.subs(c, value)) for value in (2, 10, 100)})
print("barrier minimizer x_mu =", sp.simplify(barrier_minimizer))
print("  values:", {value: round(float(barrier_minimizer.subs(mu, value)), 3) for value in (1, sp.Rational(1, 10), sp.Rational(1, 100))})

penalty minimizer x_c = c/(c + 2)
  values: {2: 0.5, 10: 0.8333333333333334, 100: 0.9803921568627451}
barrier minimizer x_mu = sqrt(2*mu + 1)/2 + 1/2
  values: {1: 1.366, 1/10: 1.048, 1/100: 1.005}


**Equivalent casadi implementation:**

In [2]:
import casadi as ca

decision_variable = ca.SX.sym("x")
objective = decision_variable**2
constraint = decision_variable

solver = ca.nlpsol("solver", "ipopt", {"x": decision_variable, "f": objective, "g": constraint},
                   {"ipopt.print_level": 0, "print_time": 0, "ipopt.sb": "yes"})
solution = solver(x0=2, lbg=1, ubg=ca.inf)

print("constrained minimizer =", float(solution["x"]))
print("minimum value =", float(solution["f"]))

constrained minimizer = 0.9999999912538532
minimum value = 0.9999999825077065


**References:**

[📘 Bertsekas, D. P. (1999). *Nonlinear Programming* (2nd ed.). Athena Scientific.](https://www.athenasc.com/nonlinbook.html)  
[📘 Nocedal, J., & Wright, S. J. (2006). *Numerical Optimization* (2nd ed.). Springer.](https://doi.org/10.1007/978-0-387-40065-5)

---

[⬅️ Previous: Augmented Lagrangian Methods](./12_augmented_lagrangian_methods.ipynb) | [Next: Integer Programming (IP) ➡️](../06_Discrete_and_Combinatorial_Optimization/01_integer_programming.ipynb)